In [1]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
from scipy.io import wavfile
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:
from huggingface_hub import snapshot_download

snapshot_download(repo_id="thucdangvan020999/singaporean_accent_district_names_continuation", 
                  repo_type="dataset", local_dir="./singaporean_accent_district_names_continuation")

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 6 files: 100%|██████████| 6/6 [00:02<00:00,  2.27it/s]


'/home/ubuntu/singaporean_accent_district_names_continuation'

In [3]:
files = glob('singaporean_accent_district_names_continuation/*/*.parquet')
files

['singaporean_accent_district_names_continuation/data/train-00000-of-00002.parquet',
 'singaporean_accent_district_names_continuation/data/train-00001-of-00002.parquet',
 'singaporean_accent_district_names_continuation/data/test-00000-of-00001.parquet',
 'singaporean_accent_district_names_continuation/data/validation-00000-of-00001.parquet']

In [4]:
df = pd.read_parquet(files[0])
df

,id,audio,audio_length_s,text,continuation,voices,district
0,Bishan_0000,{'bytes': b'RIFF\xa4\x84\x04\x00WAVEfmt \x10\x...,6.168,Bishan is well-known for its central location ...,The district of Bishan offers excellent amenit...,alloy,Bishan
1,Bishan_0001,{'bytes': b'RIFF\xa4\x80\x05\x00WAVEfmt \x10\x...,7.512,The Bishan MRT station connects residents to t...,Bishan district is well-served by these MRT li...,echo,Bishan
2,Bishan_0002,{'bytes': b'RIFF$8\x04\x00WAVEfmt \x10\x00\x00...,5.760,"Bishan Park is a green oasis in the district, ...","Located within Bishan district, the park featu...",fable,Bishan
3,Bishan_0003,{'bytes': b'RIFF$n\x04\x00WAVEfmt \x10\x00\x00...,6.048,The housing options in Bishan range from HDB f...,Bishan district is known for its well-planned ...,onyx,Bishan
4,Bishan_0004,{'bytes': b'RIFF\xa4\xa3\x03\x00WAVEfmt \x10\x...,4.968,Bishan is home to the prestigious Raffles Inst...,This district also features the scenic Bishan-...,nova,Bishan
...,...,...,...,...,...,...,...
1076,Woodlands_0078,{'bytes': b'RIFF$8\x04\x00WAVEfmt \x10\x00\x00...,5.760,The Woodlands Regional Centre is expected to g...,"Located in the northern district of Woodlands,...",onyx,Woodlands
1077,Woodlands_0079,{'bytes': b'RIFF$i\x03\x00WAVEfmt \x10\x00\x00...,4.656,The annual Woodlands Heritage Festival celebra...,"Held in the heart of the Woodlands district, t...",nova,Woodlands
1078,Woodlands_0080,{'bytes': b'RIFF\xa4\x96\x04\x00WAVEfmt \x10\x...,6.264,The Woodlands North MRT station will soon conn...,This connection will enhance accessibility for...,alloy,Woodlands
1079,Woodlands_0081,{'bytes': b'RIFF$8\x04\x00WAVEfmt \x10\x00\x00...,5.760,The Woodlands Town Garden has a picturesque la...,Located in the heart of The Woodlands district...,echo,Woodlands


In [5]:
def loop(files):

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'
    
    files, _ = files

    data = []
    for f in files:
        base = f.split('/')[0] + '_audio'
        f_new = f.replace('/', '-').replace('.parquet', '')
        os.makedirs(base, exist_ok=True)
        df = pd.read_parquet(f)
        for i in tqdm(range(len(df))):
            t = df['text'].iloc[i].strip()
            if len(t) < 2:
                continue
            audio_filename = f'{f_new}_{i}.mp3'
            audio_filename = os.path.join(base, audio_filename)
            b = df['audio'].iloc[i]['bytes']
            audio_np, sr = sf.read(io.BytesIO(b))
            if audio_np.ndim > 1:
                audio_np = audio_np.mean(axis=1)
            if audio_np.shape[0] < 10000:
                continue
            sf.write(audio_filename, audio_np, sr)
            
            data.append({
                'audio_filename': audio_filename,
                'text': t,
                'speaker': f"{base}_{df['voices'].iloc[i]}"
            })
        
    return data

In [6]:
data = multiprocessing(files, loop, cores = len(files))

100%|██████████| 1081/1081 [00:54<00:00, 19.89it/s]


In [7]:
len(data)

2543

In [8]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset[0]

{'audio_filename': 'singaporean_accent_district_names_continuation_audio/singaporean_accent_district_names_continuation-data-train-00000-of-00002_0.mp3',
 'text': 'Bishan is well-known for its central location in Singapore, making it a popular residential area',
 'speaker': 'singaporean_accent_district_names_continuation_audio_alloy'}

In [9]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'singaporean_accent_district_names_continuation')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 592.08ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (1 / 1): 100%|██████████|  138kB /  138kB,  692kB/s  
Processing Files (1 / 1): 100%|██████████|  138kB /  138kB,  345kB/s  
New Data Upload: 100%|██████████|  138kB /  138kB,  345kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:00<00:00,  1.32 shards/s]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/f476ac1b0b3a81e32b08c1a8dbd85160b1d8279c', commit_message='Upload dataset', commit_description='', oid='f476ac1b0b3a81e32b08c1a8dbd85160b1d8279c', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)

In [10]:
audio_files = [d['audio_filename'] for d in data]

with open('singaporean-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [12]:
folders = glob('singaporean_accent_district_names_continuation_audio*')
folders = [f for f in folders if '.zip' not in f]
for f in folders:
    print(f)
    os.system(f'zip -rq {f}.zip {f}')

singaporean_accent_district_names_continuation_audio
singaporean_accent_district_names_continuation_audio_neucodec


In [13]:
from huggingface_hub import HfApi
api = HfApi()

for f in glob('singaporean_accent_district_names_continuation_audio*.zip'):
    api.upload_file(
        path_or_fileobj=f,
        path_in_repo=f,
        repo_id="malaysia-ai/Multilingual-TTS",
        repo_type="dataset",
    )

Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):  38%|███▊      | 41.9MB /  111MB,   ???B/s  
Processing Files (0 / 1): 100%|█████████▉|  111MB /  111MB,  345MB/s  
Processing Files (0 / 1): 100%|█████████▉|  111MB /  111MB, 86.3MB/s  
Processing Files (1 / 1): 100%|██████████|  111MB /  111MB, 69.4MB/s  
Processing Files (1 / 1): 100%|██████████|  111MB /  111MB, 38.6MB/s  
New Data Upload: 100%|██████████|  111MB /  111MB, 38.6MB/s  
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (1 / 1): 100%|██████████| 3.16MB / 3.16MB,   ???B/s  
Processing Files (1 / 1): 100%|██████████| 3.16MB / 3.16MB,  0.00B/s  
New Data Upload: 100%|██████████| 3.16MB / 3.16MB,  0.00B/s  


In [14]:
zips = glob('*.zip')

In [16]:
for f in zips:
    if 'audio-' not in f:
        print(f)

libritts_r_filtered_clean.zip
Elise_audio_neucodec.zip
cml-tts_italian_audio_neucodec.zip
IndicTTS_Malayalam_audio_neucodec.zip
IndicTTS_Tamil_audio_neucodec.zip
cml-tts_german_audio_neucodec.zip
AISHELL3-audio.zip
IndicTTS_Bengali_audio.zip
MsceneSpeech-20250923T163109Z-1-001.zip
haqkiem-TTS_audio_neucodec.zip
cml-tts_portuguese_audio.zip
ClArTTS_audio.zip
multilingual-tts_audio.zip
cml-tts_spanish_audio.zip
singaporean_accent_district_names_continuation_audio.zip
AISHELL3-audio_neucodec.zip
multilingual-tts_audio_neucodec.zip
EmoVoice-DB_audio.zip
Elise_audio.zip
ArVoice_audio.zip
hungarian-single-speaker-tts_audio.zip
ClArTTS_audio_neucodec.zip
singlish-speaker2202_audio.zip
libritts_r_filtered_clean_neucodec.zip
cml-tts_polish_audio.zip
cml-tts_dutch_audio_neucodec.zip
MsceneSpeech-20250923T163109Z-1-002.zip
MsceneSpeech_audio_neucodec.zip
IndicTTS_Malayalam_audio.zip
cml-tts_french_audio.zip
IMDA-TTS_audio.zip
IndicTTS_Telugu_audio_neucodec.zip
kss_audio_neucodec.zip
IndicTTS_Telu